In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
import matplotlib.pyplot as plt
import numpy as np
import sentencepiece as spm
import wikipediaapi

In [2]:
import sentencepiece as spm

spm.SentencePieceTrainer.train(
    input='corpus.txt',
    model_prefix='bpe_tokenizer',
    vocab_size=200,               # 小規模な対話データに適した語彙数
    model_type='bpe',             # BPE方式
    character_coverage=1.0,       # 全文字をカバー（日本語なので1.0が安全）
    pad_id=0,
    unk_id=1,
    bos_id=2,
    eos_id=3
)


In [3]:
sp = spm.SentencePieceProcessor()
sp.load("bpe_tokenizer.model")

pad_id = sp.pad_id()
bos_id = sp.bos_id()
eos_id = sp.eos_id()


In [4]:
with open("corpus.txt", encoding="utf-8") as f:
    corpus_lines = f.readlines()


In [5]:
def make_dataset_from_corpus(sp, corpus_lines, batch_size=32, seq_len=32):
    data = []
    for _ in range(batch_size):
        line = np.random.choice(corpus_lines).strip()
        ids = sp.encode(line)
        ids = [bos_id] + ids[:seq_len - 2] + [eos_id]
        ids += [pad_id] * (seq_len - len(ids))
        data.append(ids)
    return torch.tensor(data)


In [6]:
class TransformerDecoderLayer(nn.Module):
    def __init__(self, d_model, nhead, dim_ff):
        super().__init__()
        self.self_attn = nn.MultiheadAttention(d_model, nhead, batch_first=True)
        self.linear1 = nn.Linear(d_model, dim_ff)
        self.linear2 = nn.Linear(dim_ff, d_model)
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        self.dropout = nn.Dropout(0.1)

    def forward(self, x, tgt_mask):
        attn_output, _ = self.self_attn(x, x, x, attn_mask=tgt_mask)
        x = self.norm1(x + self.dropout(attn_output))
        ff_output = self.linear2(torch.relu(self.linear1(x)))
        x = self.norm2(x + self.dropout(ff_output))
        return x

In [7]:
class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=100):
        super().__init__()
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * (-np.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        self.register_buffer('pe', pe.unsqueeze(0))  # ✅ これが重要！

    def forward(self, x):
        x = x + self.pe[:, :x.size(1)].to(x.device)
        return x

In [8]:
class MiniGPTDecoder(nn.Module):
    def __init__(self, vocab_size, d_model=64, nhead=2, dim_ff=256, max_len=100):
        super().__init__()
        self.embed = nn.Embedding(vocab_size, d_model)
        self.pos_enc = PositionalEncoding(d_model, max_len)
        self.decoder = TransformerDecoderLayer(d_model, nhead, dim_ff)
        self.out = nn.Linear(d_model, vocab_size)

    def generate_square_subsequent_mask(self, sz):
        return torch.triu(torch.full((sz, sz), float('-inf')), diagonal=1)

    def forward(self, x):
        mask = self.generate_square_subsequent_mask(x.size(1)).to(x.device)
        x = self.embed(x)
        x = self.pos_enc(x)
        x = self.decoder(x, mask)
        return self.out(x)

In [9]:
class WarmupCosineScheduler:
    def __init__(self, optimizer, d_model, warmup_steps, max_steps):
        self.optimizer = optimizer
        self.d_model = d_model
        self.warmup_steps = warmup_steps
        self.max_steps = max_steps
        self.step_num = 0

    def step(self):
        self.step_num += 1
        lr = self.get_lr()
        for param_group in self.optimizer.param_groups:
            param_group['lr'] = lr

    def get_lr(self):
        step = self.step_num
        d_model_term = self.d_model ** -0.5

        if step < self.warmup_steps:
            scale = step * (self.warmup_steps ** -1.5)
        else:
            progress = (step - self.warmup_steps) / (self.max_steps - self.warmup_steps)
            scale = 0.5 * (1 + torch.cos(torch.tensor(progress * np.pi))) / (self.warmup_steps ** 0.5)

        return d_model_term * scale

In [10]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = MiniGPTDecoder(vocab_size=sp.get_piece_size()).to(device)
optimizer = optim.Adam(model.parameters(), lr=0)
scheduler = WarmupCosineScheduler(optimizer, d_model=64, warmup_steps=20, max_steps=200)
loss_fn = nn.CrossEntropyLoss(ignore_index=pad_id)

In [11]:
losses = []
for epoch in range(200):
    model.train()
    x = make_dataset_from_corpus(sp, corpus_lines).to(device)
    x_input = x[:, :-1]
    y_target = x[:, 1:]
    logits = model(x_input)
    loss = loss_fn(logits.view(-1, logits.size(-1)), y_target.reshape(-1))
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
    scheduler.step()
    losses.append(loss.item())
    if epoch % 20 == 0:
        print(f"Epoch {epoch}, Loss: {loss.item():.4f}, LR: {optimizer.param_groups[0]['lr']:.6f}")


Epoch 0, Loss: 5.3570, LR: 0.001398
Epoch 20, Loss: 0.5992, LR: 0.027949
Epoch 40, Loss: 0.5813, LR: 0.027023
Epoch 60, Loss: 0.6010, LR: 0.024523
Epoch 80, Loss: 0.6561, LR: 0.020751
Epoch 100, Loss: 0.5564, LR: 0.016162
Epoch 120, Loss: 0.4481, LR: 0.011309
Epoch 140, Loss: 0.5036, LR: 0.006778
Epoch 160, Loss: 0.4459, LR: 0.003114
Epoch 180, Loss: 0.4586, LR: 0.000761


In [12]:
torch.save(model.state_dict(), "model.pt")
print("モデルを model.pt に保存しました。")

モデルを model.pt に保存しました。


In [13]:
def top_k_top_p_filtering(logits, top_k=0, top_p=1.0):
    sorted_logits, sorted_indices = torch.sort(logits, descending=True)
    cumulative_probs = torch.softmax(sorted_logits, dim=-1).cumsum(dim=-1)

    if top_p < 1.0:
        mask = cumulative_probs > top_p
        mask[..., 1:] = mask[..., :-1].clone()
        mask[..., 0] = 0
        sorted_logits[mask] = -float('Inf')

    if top_k > 0:
        sorted_logits[..., top_k:] = -float('Inf')

    filtered_logits = torch.full_like(logits, -float('Inf'))
    filtered_logits.scatter_(1, sorted_indices, sorted_logits)
    return filtered_logits

def generate_response(model, sp, user_input, max_len=50, temperature=1.0, top_k=0, top_p=1.0):
    model.eval()
    input_ids = [bos_id] + sp.encode(f"User: {user_input}\nAI:")
    input_tensor = torch.tensor([input_ids], device=device)

    for _ in range(max_len):
        with torch.no_grad():
            logits = model(input_tensor)
            logits = logits[:, -1, :] / temperature
            filtered_logits = top_k_top_p_filtering(logits, top_k=top_k, top_p=top_p)
            probs = torch.softmax(filtered_logits, dim=-1)
            next_token = torch.multinomial(probs, num_samples=1)

        input_tensor = torch.cat([input_tensor, next_token], dim=1)
        if next_token.item() == eos_id:
            break

    response_ids = input_tensor[0].tolist()[len(input_ids):]
    return sp.decode(response_ids)


In [14]:
history = ""  # 会話履歴（最初は空）

while True:
    user_input = input("あなた：")
    if user_input.lower() in ["exit", "quit"]:
        break

    history += f"User: {user_input}\n"

    # トークナイズ用に「履歴 + AI:」を渡す
    input_ids = [bos_id] + sp.encode(history + "AI:")
    input_tensor = torch.tensor([input_ids], device=device)

    model.eval()
    for _ in range(50):
        with torch.no_grad():
            logits = model(input_tensor)
            logits = logits[:, -1, :] / 1.0
            filtered = top_k_top_p_filtering(logits, top_k=0, top_p=0.9)
            probs = torch.softmax(filtered, dim=-1)
            next_token = torch.multinomial(probs, num_samples=1)
        input_tensor = torch.cat([input_tensor, next_token], dim=1)
        if next_token.item() == eos_id:
            break

    response_ids = input_tensor[0].tolist()[len(input_ids):]
    response = sp.decode(response_ids)

    print(f"User: {user_input}")
    print("AI：", response)

    history += f"AI: {response}\n"  # 履歴にAIの応答を追加


KeyboardInterrupt: Interrupted by user